# Shuffle video

## Import modules

In [2]:
# import internal modules
# from typing import List, Set, Dict, TypedDict, Tuple, Optional, Union
from pathlib import Path
from datetime import date
# from dataclasses import dataclass, field
# import itertools


# import 3rd-party modules
import cv2
import numpy as np

# import local modules
# from utils.renderer.giffer import create_gif
from utils.renderer.videographer import create_video
from utils.project_manager import Project
# from utils.renderer.resizer import get_interpolation
from utils.renderer.resizer import resize_with_pad, resize_with_crop

## Set up project

In [3]:
# create project
project = Project(project_dir="assets/images/shuffle")

## Shuffle list of frame indexes from video

In [4]:
# get current date
today = date.today().strftime("%Y%m%d")

# set input & output video path
video_path = Path("/Users/derrickvanfrausum/Desktop/danse_gagu.mp4")
out_path = project.project_dir / f"{video_path.stem}_{today}.mp4"

# Initialize video stream
video_cap = cv2.VideoCapture(str(video_path))

# get video parameters
video_nb_frames = int(video_cap.get(cv2.CAP_PROP_FRAME_COUNT))
video_fps = video_cap.get(cv2.CAP_PROP_FPS)
video_width = int(video_cap.get(cv2.CAP_PROP_FRAME_WIDTH))
video_height = int(video_cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

print(f"number of frames = {video_nb_frames}")
print(f"fps = {video_fps}")
print(f"video width = {video_width}")
print(f"video height = {video_height}")

# set codec for output video
codec = "H264"

rotate = False
resize = False

# set output shape
# out_height, out_width, out_channel = 1920, 1080, 3
out_height, out_width, out_channel = video_height, video_width, 3

# create a videoWriter object
fourcc = cv2.VideoWriter_fourcc(*codec)
out_video = cv2.VideoWriter(filename=str(out_path), fourcc=fourcc, fps=video_fps, frameSize=(out_width, out_height))

# get random list of frame indexes
frame_idxs = np.arange(video_nb_frames)
np.random.shuffle(frame_idxs)


# iterate over frame indexes
for frame_idx in frame_idxs:
    # set frame position to the index
    video_cap.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)

    # read video stream
    ret, frame = video_cap.read()

    if not ret:
        print(f"frame {frame_idx} is empty")
        continue

    # rotate & resize frame if asked
    if rotate:
        frame = np.rot90(frame, -1)
    
    if resize:
        frame = resize_with_crop(frame, ref_img_shape=(out_height, out_width, out_channel))

    # write output frame
    out_video.write(frame)

# release video stream & video rendering
video_cap.release()
out_video.release()

number of frames = 7293
fps = 30.004400755154407
video width = 720
video height = 1280
